# 18 — Cosine Similarity

Cosine similarity measures the angle between two vectors: cos(θ) = a·b / (|a||b|). It ranges from 1 (same direction) to -1 (opposite), with 0 meaning orthogonal. Because it normalizes by magnitude, it compares *profiles* regardless of document length.

**Why it matters for resumes / ATS:** resume–JD matching is a vector-similarity problem. Cosine gives a single 0–1 score for "how well does this resume fit this job" — the number an ATS ranks candidates by, and the same score Ch. 16's TF-IDF vectors were built to feed.

**Goal:** Measure document similarity using vector dot products.

Starting from the formula in plain numpy, this chapter builds up to a 4×4 resume similarity matrix and a resume-vs-JD scorer. Two implementation notes worth keeping: length normalization is what makes cosine fair across short and long documents, and the same score powers keyword extraction (Ch. 14) and sentence embeddings (Ch. 22).

## 1. Cosine Similarity from Scratch

The formula is three operations: dot product (shared magnitude), product of norms (scaling), and the ratio (the angle's cosine). Zero-vector inputs return 0.0 by convention — the guard `if norm > 0 else 0.0` prevents division by zero.

**What the code does:** implements `cosine_sim()` and checks three edge cases.
- Identical vectors `[1,2,3]` vs `[1,2,3]` → `1.000` — maximum similarity.
- Orthogonal `[0,0,1]` vs `[1,0,0]` → `0.000` — no shared direction.
- Opposite `[1,0]` vs `[-1,0]` → `-1.000` — anti-correlated.

**Try it:** multiply `a` by 10 and recompute — the score stays 1.000 because cosine ignores magnitude. That invariance is the entire reason it works on documents of different lengths.

In [1]:
import numpy as np
def cosine_sim(a, b):
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return dot / norm if norm > 0 else 0.0

# Two identical vectors
a = np.array([1, 2, 3])
b = np.array([1, 2, 3])
print(f"Identical: {cosine_sim(a, b):.3f}")

# Orthogonal vectors
c = np.array([0, 0, 1])
d = np.array([1, 0, 0])
print(f"Orthogonal: {cosine_sim(c, d):.3f}")

# Opposite vectors
e = np.array([1, 0])
f = np.array([-1, 0])
print(f"Opposite: {cosine_sim(e, f):.3f}")

Identical: 1.000
Orthogonal: 0.000
Opposite: -1.000


## 2. Resume Similarity with TF-IDF + Cosine

TF-IDF vectors (Ch. 16) are already length-normalized, and `cosine_similarity(X)` computes the full pairwise matrix in one call: `sim_matrix[i][j]` is the similarity between resumes i and j.

**What the code does:** vectorizes four resumes (Google/Python/NLP, Amazon/Python/ML/TF, Oracle/Java/Spring, Microsoft/MLOps), fits a 50-feature TF-IDF model, and builds the 4×4 matrix.
- Measured values: diagonal 1.000, R1↔R2 = `0.331`, R1↔R3 = `0.175`, R1↔R4 = `0.075` — the Python/ML resumes are closest, the Java resume is nearly orthogonal to the ML resumes.

**Note (known issue):** the cell's print loop formats an integer with `{i+1:15s}`, which raises `ValueError` in Python 3 — use `{i+1:<15}` to render the table. The matrix computation itself is correct.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

resumes = [
    "Python developer with NLP and machine learning experience at Google",
    "Data scientist skilled in Python, machine learning, and TensorFlow at Amazon",
    "Java backend developer with Spring Boot and microservices experience at Oracle",
    "ML engineer with TensorFlow, PyTorch, and MLOps experience at Microsoft",
]

vec = TfidfVectorizer(stop_words="english", max_features=50)
X = vec.fit_transform(resumes)
sim_matrix = cosine_similarity(X)

print("Resume Similarity Matrix:")
print(f"{'':25s}", end="")
for i in range(len(resumes)):
    print(f"R{i+1:5s}", end="")
print()
for i in range(len(resumes)):
    print(f"Resume {i+1:15s}", end="")
    for j in range(len(resumes)):
        print(f"{sim_matrix[i][j]:.3f} ", end="")
    print()

print(f"\nR1 vs R2 (both ML/Python): {sim_matrix[0][1]:.3f}")
print(f"R1 vs R3 (different stack): {sim_matrix[0][2]:.3f}")

Resume Similarity Matrix:
                         

ValueError: Unknown format code 's' for object of type 'int'

## 3. Resume vs JD Scoring

The real ATS operation: one resume, many job descriptions. Vectorize everything in the same space, take the first row of the similarity matrix, and rank the jobs by score.

**What the code does:** fits TF-IDF on `[resume, jd_ml, jd_java]` and reads row 0 of `cosine_similarity(X)`.
- Resume vs ML job: `0.510` — strong overlap on python/tensorflow/machine learning.
- Resume vs Java job: `0.000` — zero shared vocabulary after stopword removal, i.e. completely different stacks.

**Try it:** the ML job wins by a wide margin, and the 0.000 vs the Java job is a clean, explainable miss — no shared terms at all. That's cosine similarity doing its job on the exact features from Ch. 16.

In [ ]:
resume = "python tensorflow machine learning nlp data science"
jd_ml = "python machine learning tensorflow deep learning ai"
jd_java = "java spring boot microservices oracle cloud"

vec = TfidfVectorizer()
X = vec.fit_transform([resume, jd_ml, jd_java])
sim = cosine_similarity(X)
print(f"Resume vs ML job:     {sim[0][1]:.3f}")
print(f"Resume vs Java job:   {sim[0][2]:.3f}")
print("\nCosine similarity correctly identifies the better match!")

## Key Insight: Cosine similarity = angle between vectors, not magnitude. Handles different document lengths gracefully.

**Score by direction, not size: a short resume and a long JD can still align perfectly if their term profiles match.**

Magnitude normalization is what makes this fair — a 5-page CV doesn't automatically beat a 1-page one. Cosine on TF-IDF vectors is the workhorse of classical resume matching, and the exact same score is reused by KeyBERT (Ch. 14) and sentence transformers (Ch. 22). Next, Ch. 19 replaces the sparse TF-IDF vectors with dense learned embeddings.